# 07 · Un tablero que permite comprobar la evidencia

**RA3 · CE3.1–3.4.** Dos herramientas concretas: Python y Excel; receta adicional Power BI. Una sesión de construcción y otra de comunicación.

**Caso:** ComercioSur, datos simulados. Ejecutar en orden con kernel activo. Cada ejercicio exige una interpretación y un límite; consultar [manual](../material_propio/EBA_manual_cientifico.html) y [guía](../guia_maestra_eba.html). El curso presupone manejo elemental de tablas de Fundamentos. No instalar paquetes desde las celdas.

### Antes de ejecutar

Núcleo · 90 min guiados + 45 min autónomos orientativos.

**Objetivo:** Conciliar un indicador entre herramientas y diseñar un tablero interpretable.

**Preparación:** Agregaciones, filtros y conclusiones de los módulos anteriores.

**Ejemplo de referencia:** Comparar los agregados por canal y campaña en Python y el libro Excel.

**Práctica:** Cambiar un filtro, revisar población y denominador y explicar la misma medida en DAX.

**Criterio de logro:** Entregar un tablero con unidades, filtros y totales conciliados; identificar qué se ejecutó.

**Distribución orientativa:** explicación 30 min; práctica guiada 60 min; trabajo autónomo 45 min. Ajustar tras una clase piloto.

[Guía y secuencia](../guia_maestra_eba.html#ruta-aprendizaje) · [Fundamento del manual](../material_propio/EBA_manual_cientifico.html#sec-visualizacion)

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, Markdown

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "_transversal/datos").is_dir())
sys.path.insert(0, str(RAIZ / "02_Estadistica_Business_Analytics/reproducibilidad"))
from eba_recursos import *

df = pedidos()
plt.rcParams.update(
    {"figure.figsize": (7, 3.4), "axes.spines.top": False, "axes.spines.right": False}
)

**Práctica importable de BI:** [instrucciones, consulta M, medidas DAX y controles](../material_propio/herramientas_bi/README.md).
La conciliación con Python está disponible; el alumno documenta importación, filtros y resultados del motor que utiliza.

<a id="herramientas"></a>
## Elegir herramientas con criterio

Python facilita pruebas y reproducción; Excel, inspección por usuarios de negocio; Power BI, modelo y filtros compartidos. Compare acceso, actualización, trazabilidad y experiencia de audiencia. No elegir herramienta solo por estética.

In [2]:
display(
    pd.DataFrame(
        {
            "herramienta": ["Python", "Excel", "Power BI"],
            "fortaleza": [
                "Código y pruebas",
                "Inspección y fórmulas",
                "Filtros y modelo relacional",
            ],
            "límite": [
                "Requiere entorno",
                "Cambios manuales difíciles de auditar",
                "Requiere instalación/licencia según uso",
            ],
        }
    )
)

,herramienta,fortaleza,límite
0,Python,Código y pruebas,Requiere entorno
1,Excel,Inspección y fórmulas,Cambios manuales difíciles de auditar
2,Power BI,Filtros y modelo relacional,Requiere instalación/licencia según uso


<a id="dashboard"></a>
## Dos preguntas visuales distintas

Distribución del tiempo y comparación de medias por campaña. Mostrar tamaño por grupo e IC; el promedio global puede esconder diferencias de composición. Los filtros cambian población y precisión. No usar búsqueda exploratoria de segmentos como prueba confirmatoria.

In [3]:
@widgets.interact(canal=widgets.Dropdown(options=["Todos", "Web", "Tienda"]))
def tablero(canal):
    f = df if canal == "Todos" else df[df.canal.eq(canal)]
    display(indicadores(f))
    res = pd.DataFrame([{"campana": g, **ic_media(x.tiempo_min)} for g, x in f.groupby("campana")])
    display(res[["campana", "n", "media", "li", "ls"]])
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].hist(f.tiempo_min, bins=18)
    ax[0].set(xlabel="Tiempo (min)", ylabel="Pedidos")
    ax[1].errorbar(
        res.campana, res.media, yerr=[res.media - res.li, res.ls - res.media], fmt="o", capsize=5
    )
    ax[1].set(xlabel="Campaña", ylabel="Media e IC 95% (min)")
    plt.show()

interactive(children=(Dropdown(description='canal', options=('Todos', 'Web', 'Tienda'), value='Todos'), Output…

<a id="excel"></a>
## Construir un libro de consulta y fórmulas

El archivo se genera con tablas de datos, resumen precalculado, fórmulas y un gráfico. openpyxl escribe fórmulas; se agregan valores cacheados calculados en Python para lectura sin motor de cálculo. Esto no ejecuta Excel: al abrir en Excel se deben recalcular y contrastar con la hoja de valores. Esta sesión no automatiza Excel de escritorio.

In [4]:
from openpyxl import Workbook, load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.workbook.properties import CalcProperties

SALIDA = RAIZ / "salidas"
SALIDA.mkdir(exist_ok=True)  # copia de trabajo; la versión publicada está en material_propio
wb = Workbook()
ws = wb.active
ws.title = "Pedidos"
ws.append(df.columns.tolist())
for row in df.itertuples(index=False, name=None):
    ws.append(list(row))
t = Table(displayName="Pedidos", ref=f"A1:F{len(df)+1}")
t.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
ws.add_table(t)
ws.freeze_panes = "A2"
res = wb.create_sheet("Resumen_valores")
res.append(["Campana", "n", "Tiempo_medio_min", "Ticket_medio_mil", "Resueltos_pct"])
for g, x in df.groupby("campana"):
    res.append([g, len(x), x.tiempo_min.mean(), x.ticket_mil.mean(), 100 * x.resuelto.mean()])
form = wb.create_sheet("Formulas")
form.append(["Campana", "n", "Tiempo_medio_min"])
for i, g in enumerate(["A", "B", "C"], 2):
    form.append(
        [g, f"=COUNTIF(Pedidos!C2:C601,A{i})", f"=AVERAGEIF(Pedidos!C2:C601,A{i},Pedidos!E2:E601)"]
    )
chart = BarChart()
chart.title = "Tiempo por campaña · datos simulados"
chart.y_axis.title = "Minutos"
chart.add_data(Reference(res, min_col=3, min_row=1, max_row=4), titles_from_data=True)
chart.set_categories(Reference(res, min_col=1, min_row=2, max_row=4))
res.add_chart(chart, "G2")
wb.calculation = CalcProperties(fullCalcOnLoad=True)
path = SALIDA / "EBA_tablero_excel.xlsx"
wb.save(path)
from _transversal.herramientas import guardar_cache_formulas

cache_excel = {"Formulas": {}}
for fila_excel, campana_excel in enumerate(["A", "B", "C"], 2):
    grupo_excel = df.loc[df.campana.eq(campana_excel)]
    cache_excel["Formulas"][f"B{fila_excel}"] = len(grupo_excel)
    cache_excel["Formulas"][f"C{fila_excel}"] = grupo_excel.tiempo_min.mean()
guardar_cache_formulas(path, cache_excel)
print("Libro guardado:", path.name)

Libro guardado: EBA_tablero_excel.xlsx


<a id="conciliacion"></a>
## Contrastar herramientas

Verifique 600 registros y agregados contra pandas. En Excel convierta los datos en tabla dinámica con campaña en filas, promedio de tiempo y recuento de id; filtre canal y contraste con el widget. Una tabla sin n ni unidad no basta.

In [5]:
libro = load_workbook(path, data_only=True)
assert libro["Pedidos"].max_row == 601
for row in libro["Resumen_valores"].iter_rows(min_row=2, values_only=True):
    g, n, tiempo, ticket, resuelto = row
    f = df[df.campana.eq(g)]
    assert n == len(f)
    assert np.isclose(tiempo, f.tiempo_min.mean())
    assert np.isclose(ticket, f.ticket_mil.mean())
    assert np.isclose(resuelto, 100 * f.resuelto.mean())
assert load_workbook(path, data_only=False)["Formulas"]["B2"].value.startswith("=COUNTIF")
print(
    "Filas, agregados y fórmulas escritas verificados; recálculo en Excel pendiente del usuario del libro."
)

Filas, agregados y fórmulas escritas verificados; recálculo en Excel pendiente del usuario del libro.


<a id="powerbi"></a>
## Receta reproducible para Power BI

Importar comerciosur_pedidos.csv, id entero, ticket/tiempo decimal y resuelto entero. Crear medidas DAX `Pedidos = COUNTROWS(Pedidos)`, `Tiempo medio = AVERAGE(Pedidos[tiempo_min])`, `Resueltos % = DIVIDE(SUM(Pedidos[resuelto]), [Pedidos])`. Formatear esta última como porcentaje, segmentador canal y barras campaña. Resultado esperado: mismo n y medias que Python, tanto global como por filtro. Es una receta para practicar; no se entrega un PBIX ni se afirma haber ejecutado Power BI.

In [6]:
display(
    pd.DataFrame(
        [
            {"canal": c, **indicadores(df if c == "Todos" else df[df.canal.eq(c)])}
            for c in ["Todos", "Web", "Tienda"]
        ]
    )
)

,canal,n,ticket_medio,tiempo_medio,resueltos_pct
0,Todos,600,50.913017,36.256283,75.500000
1,Web,383,48.059478,37.562689,74.673629
2,Tienda,217,55.949447,33.950507,76.958525


<a id="impacto"></a>
## Evaluar si la visualización mejora una decisión

Actividad: compare dos prototipos del mismo resultado, tabla y gráfico de puntos con IC. Asigne versiones aleatoriamente a lectores, mantenga datos/instrucciones constantes y mida aciertos, tiempo y confianza calibrada; defina éxito antes. Número de vistas no mide calidad de decisión.

In [7]:
protocolo = {
    "pregunta": "¿Qué campaña reduce tiempo con incertidumbre aceptable?",
    "resultado_primario": "Proporción de respuestas correctas con justificación",
    "secundario": "Tiempo de respuesta en segundos",
    "asignacion": "Aleatoria entre versiones A/B",
    "criterio": "Más aciertos sin sobreconfianza; umbral predefinido",
    "limite": "Piloto pequeño no prueba impacto comercial",
}
display(pd.Series(protocolo))

pregunta              ¿Qué campaña reduce tiempo con incertidumbre a...
resultado_primario    Proporción de respuestas correctas con justifi...
secundario                              Tiempo de respuesta en segundos
asignacion                                Aleatoria entre versiones A/B
criterio              Más aciertos sin sobreconfianza; umbral predef...
limite                       Piloto pequeño no prueba impacto comercial
dtype: str

## Comprobación y entrega

Responda la autoevaluación y complete la actividad del último bloque. Entregue objetivo, método, supuesto, resultado con unidades y decisión condicionada. Las soluciones orientadoras permiten estudiar, pero no sustituyen su interpretación.

In [8]:
auto_07 = pregunta(
    "¿Un dashboard bonito demuestra que mejoró las decisiones?",
    "Sí, el diseño basta",
    "No, hay que evaluar comprensión y decisiones con un protocolo",
    "Mida aciertos y tiempo manteniendo constante la información; separe usabilidad de impacto comercial.",
    tercera="Basta con que todos los indicadores mejoren después de instalarlo",
)
display(auto_07)

## Práctica de transferencia 07

Filtra Tienda en ComercioSur y concilia cantidad, tiempo medio y resolución en Python, Excel y la consulta Power Query entregada. Diseña una tarea de usuario que evalúe comprensión del tablero manteniendo iguales los datos.

**Entrega:** hipótesis previa, cálculo con unidades, interpretación y límite. Completa tu respuesta antes de consultar la [pauta docente](../material_propio/PAUTA_DOCENTE.md#nb07).

**Discusión:** ¿Cuál es la diferencia entre una fórmula escrita y una fórmula recalculada por Excel?

### Tu resolución

Edita esta celda: escribe tu hipótesis, procedimiento, resultado, interpretación y una comprobación. Adjunta tu código o gráfico si la actividad lo requiere.

**Auto-revisión:** ¿usé la población correcta?, ¿declaré unidades y supuestos?, ¿mi evidencia permite esa conclusión?